## Common/repeated genes

In [1]:
import pandas as pd
import os
def prepare_ranking(csv_file):
    # Load the CSV file containing Ensembl names, symbols, and ranks
    #csv_file = "/home/karen/Documents/GitHub/Identify_muscle_age_genes/deep_learning/Results/feature_selection/RNAseq/ridge_L2_0/Experiment_GSE60590_feature_selection_Symbols.csv"
    gene_rank_df = pd.read_csv(csv_file)
    # check the first rows of the dataframe, if it starts with "","coef","abs_coef","NULL", replace NULL with "Symbol" and "" with "Ensembl"
    gene_rank_df.columns = ["Ensembl","coef","abs_coef","Symbol"]
    if gene_rank_df.columns[0] == "V1":
        gene_rank_df.columns = ["Ensembl","coef","abs_coef","Symbol"]
        # remove first row
        gene_rank_df = gene_rank_df.iloc[1:]
    # check if the first row value is NaN
    if gene_rank_df.iloc[0][1] == "coef":
        gene_rank_df = gene_rank_df.iloc[1:]
    
    gene_rank_df = gene_rank_df.fillna(0)
    gene_rank_df["abs_coef"] = pd.to_numeric(gene_rank_df["abs_coef"], errors='coerce')
    gene_rank_df = gene_rank_df[gene_rank_df["abs_coef"]>0.0488]
    return gene_rank_df

In [2]:

csv_path = "Results/feature_selection/RNAseq/ridge_L2_0/"
file_list = os.listdir(csv_path)

In [3]:
file_list

['Experiment_GSE157585_feature_selection.csv',
 'Union',
 'Experiment_GSE152558_feature_selection.csv',
 'GSEA',
 'Experiment_GSE129643_feature_selection_Symbols.csv',
 'Experiment_GSE167186_feature_selection_Symbols.csv',
 'Status_trained_feature_selection.csv',
 'Experiment_GSE152558_feature_selection_Symbols.csv',
 'Sex_male_feature_selection_Symbols.csv',
 'Status_Sarcopenia_feature_selection_Symbols.csv',
 'Experiment_GSE164471_feature_selection.csv',
 'Sex_male_feature_selection.csv',
 'Status_Healthy_feature_selection_Symbols.csv',
 'Sex_female_feature_selection.csv',
 'Experiment_GSE129643_feature_selection.csv',
 'GO_Biological_Process_2018',
 'Status_Healthy_feature_selection.csv',
 'Sarcopenia_only',
 'readme.txt',
 'Experiment_GSE60590_feature_selection.csv',
 'Status_Sarcopenia_feature_selection.csv',
 'Experiment_GSE60590_feature_selection_Symbols.csv',
 'set_sarcopenia_genes.txt',
 'Experiment_GSE157585_feature_selection_Symbols.csv',
 'Experiment_GSE164471_feature_selec

In [4]:
sets_genes = []
sets_ranks = {}
ranks_sarcopenia={}
set_sarcopenia= set()
## for file in file_list, get the first column, get the values
for file in file_list:
    if file.endswith("_Symbols.csv"):
        df = prepare_ranking(csv_path+file)
        # get the first column
        genes = df["Symbol"].values
        sets_genes.append(set(genes))
        # add all the genes to the dictionary with the abs_coef as the value, if it already exist, sum the values
        for gene, coef in zip(df["Symbol"], df["abs_coef"]):
            if gene in sets_ranks:
                sets_ranks[gene] += coef
            else:
                sets_ranks[gene] = coef
        
        if "Sarcopenia" in file:
            set_sarcopenia = set(genes)
            for gene, coef in zip(df["Symbol"], df["abs_coef"]):
                ranks_sarcopenia[gene] = coef


#sets_genes

/tmp/ipykernel_81067/348435330.py:14: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  if gene_rank_df.iloc[0][1] == "coef":
/tmp/ipykernel_81067/348435330.py:14: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  if gene_rank_df.iloc[0][1] == "coef":
/tmp/ipykernel_81067/348435330.py:14: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  if gene_rank_df.iloc[0][1] == "coef":
/tmp/ipykernel_81067/348435330.py:14: FutureWarning: Series.__getitem_

In [5]:
len(sets_genes)

11

In [6]:
# remove empty sets
sets_genes = [x for x in sets_genes if x]


In [7]:
# get the intersection of all sets
intersection = set.intersection(*sets_genes)
intersection


{0}

In [8]:
for s in sets_genes:
    print(len(s))

797
307
1806
3290
327
2701
706


In [9]:
# union all genes
union = set.union(*sets_genes)

In [10]:
len(union)

5005

In [11]:
len(set_sarcopenia)

0

In [15]:

set_sarcopenia
# save the set_sarcopenia and union to a file
with open("Results/feature_selection/RNAseq/ridge_L2_0/set_sarcopenia_genes.txt", "w") as f:
    for gene in set_sarcopenia:
        f.write(str(gene)+"\n")

In [13]:
# get elements that apear in at least half of the sets
half_intersection = set()
for gene in union:
    count = 0
    for genes in sets_genes:
        if gene in genes:
            count += 1
    if count >= 8:
        half_intersection.add(gene)
len(half_intersection)

17467

In [14]:
len(sets_ranks)

25019

In [15]:
sets_ranks
# order the dictionary by value and change the value for a rank from 1 to len(sets_ranks)
sets_ranks = {k: v for k, v in sorted(sets_ranks.items(), key=lambda item: item[1], reverse=True)}
# asing the value to the dictionary on the position they have
for i, gene in enumerate(sets_ranks):
    sets_ranks[gene] = i+1


In [16]:
len(sets_ranks)

25019

In [17]:
len(intersection)

10545

In [18]:
sets_ranks.keys()


dict_keys([0, 'NDST2', 'SLC49A4', 'FAM153B', 'LSP1P5', 'RUNX1T1', 'IFNAR2', 'PRH1', 'CLN3', 'COMMD7', 'RBM27', 'C4orf36', 'GTPBP6', 'TPPP2', 'ULK4P1', 'ST6GALNAC6', 'ARL16', 'ASB15', 'TEKT3', 'LRTOMT', 'GNGT2', 'ARHGAP11B', 'RPSAP64', 'ADPRH', 'GOLGA6L9', 'DARS1', 'TRMT44', 'CARD10', 'TMC8', 'CAPN8', 'MYL12B', 'COMT', 'RPL9P30', 'RAPGEF3', 'SLC4A1', 'EIF2S3', 'TBC1D3', 'TSPYL4', 'TRMT9B', 'LVRN', 'CLSTN3', 'PHF7', 'PHLPP1', 'FBXO10', 'NIPAL3', 'PLEKHM1', 'PARP11', 'C3orf14', 'C11orf53', 'BTN3A2', 'CST3', 'FLVCR2', 'DET1', 'C8orf33', 'CEP95', 'RPL36AP18', 'TMCO1', 'PROSER1', 'IRF9', 'SPCS3', 'PEDS1', 'GORAB', 'ARMC1', 'PECAM1', 'ATP6V0B', 'ATP5MC1', 'PPP2R2B', 'SPAG1', 'VWCE', 'MCAT', 'PRB2', 'HMGN1P8', 'EIF4EBP1', 'CIDECP1', 'DPY19L3', 'SUGT1P3', 'ARSK', 'ZNF800', 'ZNF101', 'MLKL', 'MFNG', 'STRN4', 'SEC23A', 'SLC22A5', 'DPY19L1', 'MYO9B', 'KCTD21', 'CDO1', 'ARHGAP25', 'ASB13', 'PDE4C', 'PSMD14', 'RPL21P51', 'CYCSP3', 'GPR82', 'UVSSA', 'CDH18', 'ARHGEF1', 'PRPF19', 'YWHAQP5', 'PI4KAP2',

In [107]:
import GSEA_suit as gsea
folder_save = "Results/feature_selection/RNAseq/ridge_L2_0/Union/"
results = gsea.get_GSEA(rank_dict=sets_ranks, database= "GO_Biological_Process_2023", min_size=5, outdir=folder_save)


In [80]:
enrichment = results.res2d
enrichment[enrichment['FDR q-val']<0.5]

,Name,Term,ES,NES,NOM p-val,FDR q-val,FWER p-val,Tag %,Gene %,Lead_genes
4,prerank,Monocyte Chemotaxis (GO:0002548),0.60337,1.749518,0.0,0.460129,0.694,8/8,39.71%,CCL2;FOLR2;PTPRO;PDGFB;DEFB104A;CCL26;RPS19;FLT1
5,prerank,Regulation Of Chromosome Separation (GO:1905818),0.66657,1.747433,0.009901,0.349077,0.701,7/7,33.39%,NCAPD3;NCAPD2;NUMA1;NCAPG2;NCAPH2;CSNK2A2;TPR
6,prerank,Nucleotide-Sugar Metabolic Process (GO:0009225),0.534091,1.699603,0.0,0.394311,0.817,9/9,46.64%,SLC35A3;GFPT1;ENTPD5;FUT8;GALT;PARG;UGP2;EXTL2...
7,prerank,Positive Regulation Of Chromosome Separation (...,0.71082,1.675042,0.025478,0.389301,0.86,5/5,28.95%,NCAPD3;NCAPD2;NUMA1;NCAPG2;NCAPH2
8,prerank,Regulation Of Sister Chromatid Cohesion (GO:00...,0.693011,1.671465,0.006061,0.342928,0.865,5/5,30.73%,CTCF;BUB1;DDX11;FEN1;WAPL
9,prerank,Positive Regulation Of Chromatin Binding (GO:0...,0.68839,1.657586,0.01875,0.32768,0.884,5/5,31.19%,PARP9;DDX11;MED9;ZNF618;GMNN
11,prerank,piRNA Processing (GO:0034587),0.593684,1.613513,0.0,0.396544,0.945,7/7,40.67%,MOV10L1;PIWIL4;TDRD9;GPAT2;TEX15;TDRKH;DDX4
19,prerank,Protein Localization To Cell-Cell Junction (GO...,0.541647,1.56201,0.0,0.493436,0.983,8/8,45.88%,SCRIB;ACTG1;PAK2;TJP1;TJP3;LSR;JAK1;ARHGEF18


In [ ]:
# get all the genes in set_sarcopenia from sets_ranks
sarcopenia_scores={}
i = 1
for gene in set_sarcopenia:
    if gene != 0:
        sarcopenia_scores[gene] = int(sets_ranks[gene])
        #i+=1

#sorted_sarcopenia = sorted(sarcopenia_scores.items(), key=lambda x:x[1])
#sarcopenia_scores = dict(sorted_sarcopenia)

#sarcopenia_scores= sorted(sarcopenia_scores)
sarcopenia_scores




{'SCAPER': 57,
 'RPL24': 2277,
 'RPRD1B': 1019,
 'EFNA5': 6093,
 'DYNLL1': 966,
 'NPLOC4': 4973,
 'TRAPPC6B': 10335,
 'GCC1': 8075,
 'KIF5B': 4165,
 'CD36': 565,
 'UCP3': 1392,
 'SRSF3': 4274,
 'ZC3H18': 5118,
 'KHDRBS3': 584,
 'LAP3': 2158,
 'CAAP1': 4836,
 'STARD10': 4922,
 'SMAD9': 3311,
 'RPL18': 1834,
 'CHURC1': 1449,
 'RER1': 686,
 'RAD17': 12545,
 'HMBOX1': 901,
 'FN1': 4000,
 'FIBP': 2954,
 'RBM12B': 3554,
 'ANGPTL1': 5003,
 'ETV6': 5376,
 'STX7': 1422,
 'RNF31': 4669,
 'CIDEB': 5756,
 'TMEM108': 2266,
 'HERC4': 4022,
 'CDC37': 4539,
 'RGMA': 7621,
 'TADA2A': 7870,
 'PACS1': 2249,
 'MRPS28': 5963,
 'POLR2G': 3019,
 'CASK': 4927,
 'ATP2A2': 1520,
 'MAP7D1': 3354,
 'CLPX': 6656,
 'MZT2B': 5927,
 'MAN2A1': 2100,
 'DCTPP1': 5398,
 'KDM3B': 1548,
 'SAP30BP': 3818,
 'NDUFS6': 3857,
 'MYH14': 4245,
 'CHD6': 2325,
 'TMOD4': 4316,
 'EYA4': 2166,
 'GART': 5702,
 'COL1A2': 1727,
 'RSPO3': 70,
 'KALRN': 2120,
 'RABEP2': 497,
 'RAB2B': 10612,
 'AURKAIP1': 5913,
 'DUSP13': 2604,
 'CHMP3': 59

In [55]:
folder_save = "Results/feature_selection/RNAseq/ridge_L2_0/Sarcopenia_only/"
results = gsea.get_GSEA(rank_dict=ranks_sarcopenia, database= "GO_Biological_Process_2023", min_size=10, outdir=folder_save)
enrichment = results.res2d
enrichment[enrichment['FDR q-val']<0.5]

,Name,Term,ES,NES,NOM p-val,FDR q-val,FWER p-val,Tag %,Gene %,Lead_genes
0,prerank,Regulation Of Transcription By RNA Polymerase ...,-0.416095,-1.883857,0.016077,0.040161,0.032313,9/15,22.43%,DNAJB1;SMAD9;CBFB;ETV6;APBB2;HMBOX1;HDAC9;CHD6...
1,prerank,Regulation Of DNA-templated Transcription (GO:...,-0.333359,-1.424825,0.086567,0.099398,0.139456,7/13,22.43%,SMAD9;CBFB;ETV6;APBB2;HMBOX1;HDAC9;TADA2A


In [ ]:
enrichment

,Name,Term,ES,NES,NOM p-val,FDR q-val,FWER p-val,Tag %,Gene %,Lead_genes
0,prerank,Pathways in cancer,0.602273,1.407538,0.075775,1.0,0.552,6/7,39.04%,TPM3;ADCY2;SOS1;RALB;GSTP1;PIK3CA
1,prerank,Phospholipase D signaling pathway,0.593407,1.261982,0.162651,1.0,0.801,5/5,42.25%,ADCY2;SOS1;RALB;PIK3CA;DGKD
2,prerank,Cardiac muscle contraction,0.553141,1.246202,0.190092,1.0,0.829,5/6,40.64%,TPM3;COX2;TNNC1;MYL2;UQCRH
3,prerank,Dilated cardiomyopathy (DCM),0.552211,1.174796,0.241718,0.989676,0.917,4/5,39.57%,TPM3;ADCY2;TNNC1;MYL2
4,prerank,Adrenergic signaling in cardiomyocytes,0.552211,1.174796,0.241718,0.989676,0.917,4/5,39.57%,TPM3;ADCY2;TNNC1;MYL2
5,prerank,Thermogenesis,0.49645,1.167662,0.246085,0.851081,0.924,3/7,13.90%,ADCY2;SOS1;COX2
6,prerank,Hepatitis B,0.526488,1.12439,0.297101,0.876644,0.952,3/5,18.72%,SOS1;NFATC3;DDB1
7,prerank,MAPK signaling pathway,0.505906,1.074385,0.352871,0.923692,0.97,3/5,35.29%,SOS1;NFATC3;FLNC
8,prerank,Focal adhesion,0.431552,1.050279,0.407572,0.893425,0.981,5/8,39.57%,SOS1;FLNC;MYLK3;PIK3CA;MYL2
9,prerank,cGMP-PKG signaling pathway,0.446508,0.952337,0.527578,1.0,0.995,2/5,13.37%,ADCY2;NFATC3
